# Step 2.10: a backtest in one call, its artifacts, and its key

This notebook is the check of step 2.10. `ff.backtest.run` backtests a signal strategy in one call: it loads the strategy's bars, evaluates its signals with Polars, pins the store's files it reads, builds the engine's cache of those months, runs the Rust engine through its Python bindings, and saves the run's tables and its record, `run.json`, in a folder named by the run's key under `FEFAT_HOME/workspaces/default/artifacts/runs/`. A second call with the same strategy, window, and settings has the same key, so it returns the saved run without running the engine.

The notebook runs the first backtest's strategy (step 2.2) on the same window of your store's 1-minute bars as `notebooks/02-first-backtest.ipynb`, the last 23 sessions of the contract that was the front longest in the last 120 days, once through `ff.backtest.run` and once through the reference engine, both filling on the 1-minute bars and paying the fees of `retail-default`, and compares the two trade lists. Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and writes one run, and its engine cache, under `FEFAT_HOME`.

## The window

The product and the window `notebooks/02-first-backtest.ipynb` takes: MES when your store holds its 1-minute bars, and the last 23 sessions of the contract that was the front longest in the last 120 days, so that no roll falls inside the window.

Expected: one line such as `MES from databento: MESU2026, 23 sessions from 2026-08-21 to 2026-09-23`.

In [ ]:
import os
import time
from datetime import time as clock
from datetime import timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.backtest import RunConfig
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.fees import TABLES
from fefat.market.saved_instruments import instrument_table
from fefat.market.schema import Timeframe
from fefat.reference.costs import RunFees
from fefat.reference.engine import ReferenceEngine
from fefat.reference.hours import CalendarHours
from fefat.reference.ledger import trades_of
from fefat.reference.settings import RunSettings
from fefat.reference.strategy import decisions_of
from fefat.reference.tape import tape_of_bars
from fefat.reference.template import TemplateStrategy
from fefat.reference.units import instrument_parameters

paths = resolve_paths(os.environ)
instruments = instrument_table(paths.data_dir)
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
daily = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=120), LAST, fidelity="1m", provider=PROVIDER
).collect()
fronts = (
    daily.group_by("contract")
    .agg(sessions=pl.len(), last=pl.col("trade_date").max())
    .sort("sessions", "last", descending=True)
)
CONTRACT = fronts.item(0, "contract")
held = daily.filter(pl.col("contract") == CONTRACT).get_column("trade_date").sort().tail(23)
START, END = held.min(), held.max()
print(f"{ROOT} from {PROVIDER}: {CONTRACT}, {counted(held.len(), 'session')} from {START} to {END}")

## The first backtest through `ff.backtest`

The strategy of step 2.2 as `notebooks/checks/2.03.ipynb` writes it: a long when the 5-minute close crosses above the mean of the last 20 closes between 08:30 and 15:00 in the product's time zone, a stop of 12 ticks and a target of 24, and out at the first close from 15:05. The run fills on the 1-minute bars (`fidelity="1m"`) and pays the fees of `retail-default`; every other setting is the default, which the run's record writes in.

Expected: the run's time, a few seconds at most, its fills, trades, and net profit and loss, the folder of its artifacts, its key, and the first trades. A warning names the fee table when its first schedule starts after the window (F-FEE-01).

In [ ]:
instrument = instruments.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
local_close = (
    pl.from_epoch("ts_close", time_unit="ns")
    .dt.replace_time_zone("UTC")
    .dt.convert_time_zone(zone)
    .dt.time()
)
cross_up = (pl.col("close") > pl.col("mean")) & (
    pl.col("close").shift(1) <= pl.col("mean").shift(1)
)
in_hours = pl.col("local_close").is_between(clock(8, 30), clock(15, 0), closed="left")
first_backtest = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"mean": pl.col("close").rolling_mean(20), "local_close": local_close},
    long_entry=cross_up & in_hours,
    short_entry=pl.lit(False),
    long_exit=pl.col("local_close") >= clock(15, 5),
    execution=ff.Execution(
        entry=ff.Market(), stop=ff.Stop(ticks=12), targets=[ff.Target(ticks=24)], sizing=ff.Fixed(1)
    ),
)
config = RunConfig(fidelity="1m", fee_table="retail-default", provider=PROVIDER)
started = time.perf_counter()
run = ff.backtest.run(first_backtest, START, END, config)
SECONDS = time.perf_counter() - started
summary = run.record.summary
print(f"Run: {SECONDS:.1f} s, {summary.fills} fills, {summary.trades} trades, net {summary.pnl} {instrument.currency}")
print(f"Artifacts: {run.folder}")
print(f"Key: {run.hash}")
display(run.trades.select("entry_ts", "exit_ts", "exit_reason", "gross_pnl", "fees", "pnl").head())

## The reference engine's trade list

The same strategy through the reference engine, as `notebooks/checks/2.03.ipynb` runs it: the contract's own 1-minute bars as the tape, its 5-minute bars as the signal bars, and the same fee table.

Expected: the number of trades the reference engine makes.

In [ ]:
sessions = instruments.sessions_of(ROOT, paths.cache_dir)
parameters = instrument_parameters(instrument)
one_minute = Timeframe.parse("1m")
five_minutes = ff.market.load(CONTRACT, "5m", START, END, fidelity="1m", provider=PROVIDER)
minutes = ff.market.load(CONTRACT, "1m", START, END, fidelity="1m", provider=PROVIDER).collect()
tape = tape_of_bars(minutes)
hours = CalendarHours.around(sessions, tape[0].starts_at(), tape[-1].available_at())
signals = first_backtest.signals(five_minutes).collect()
template = TemplateStrategy(first_backtest.to_spec(), signals, parameters, sessions)
fees = RunFees.of_table(TABLES["retail-default"], ROOT)
engine = ReferenceEngine(RunSettings.for_fidelity(one_minute), parameters, one_minute, hours, fees=fees)
REFERENCE = trades_of(engine.run(tape, decisions_of(signals), template.on_bar).fills)
print(f"Reference engine: {counted(len(REFERENCE), 'trade')}")

## The two trade lists

Each trade's entry and exit, their prices, the exit reason, the profit and loss before fees, and the fees, from both runs. A trade that only one of them makes is a difference, listed in the table below the counts.

Expected: the same number of trades from both, then `Differences: 0` and an empty table.

In [ ]:
columns = ["entry_ts", "entry_price", "exit_ts", "exit_price", "exit_reason", "gross_pnl", "fees"]
mine = run.trades.sort("exit_ts").select(columns)
theirs = pl.DataFrame(
    [
        {
            "entry_ts": trade.entry_ts,
            "entry_price": trade.entry_price,
            "exit_ts": trade.exit_ts,
            "exit_price": float(trade.exit_price),
            "exit_reason": str(trade.exit_reason),
            "gross_pnl": trade.pnl,
            "fees": trade.fees,
        }
        for trade in REFERENCE
    ],
    schema=mine.schema,
)
DIFFERENCES = pl.concat(
    [
        mine.join(theirs, on=columns, how="anti").with_columns(only_in=pl.lit("ff.backtest")),
        theirs.join(mine, on=columns, how="anti").with_columns(only_in=pl.lit("the reference")),
    ]
)
print(f"Trades: {mine.height} through ff.backtest, {theirs.height} through the reference engine")
print(f"Differences: {DIFFERENCES.height}")
display(DIFFERENCES)

## The same run again

The same strategy, window, and settings have the same key, so the second call reads the saved run back without running the engine. The key is worked out from the run's signal frame and from the store's files it reads, so both calls load the bars and evaluate the signals, which takes most of either call's time.

Expected: about the first call's time, and `the same key: True`.

In [ ]:
started = time.perf_counter()
again = ff.backtest.run(first_backtest, START, END, config)
print(f"Second call: {time.perf_counter() - started:.1f} s, the same key: {again.hash == run.hash}")

## Your check

Run `notebooks/checks/2.10.ipynb`: the same strategy as 2.2 runs on the full month in seconds and the trade list equals the reference engine's list from 2.2 (the notebook compares them).

The run's time is the first code cell's `Run:` line, and the comparison is `Differences: 0`.